## 基于MindNLP的Vision Transformer模型应用开发

**环境配置：**

1. MindSpore 2.3.1
2. Mindnlp 0.4.0
3. Python 3.9

### 1 Vision Transformer模型
原文地址：https://arxiv.org/abs/2010.11929

代码链接：https://github.com/google-research/vision_transformer

#### 1.1 ViT 模型

Vision Transformer (ViT) 基本上可以看作是BERT的图像版本。它在与最先进的卷积神经网络相比时取得了优异的结果。需要注意的是，已经有一些改进方法出现（例如Facebook AI推出的[DeiT](https://ai.facebook.com/blog/data-efficient-image-transformers-a-promising-new-technique-for-image-classification/) = 数据高效图像Transformer），而这些改进方法也已经移植到了HuggingFace Transformers中。

每张图像都会被划分为一系列不重叠的patch（如16x16或32x32的分辨率），并进行线性嵌入。这在某些情况下也被称为卷积操作 😉。接下来，绝对位置嵌入被加入，并传递给一系列编码器层。在此过程中，会在开头添加一个[CLS]标记，以获得整个图像的全局表示。在最后的隐藏状态上可以添加一个线性分类头，用于对图像进行分类。

#### 1.2 导入模型到mindnlp
可以从中 `dir(mindnlp.transformers)` 查看和导入到关于VIT的包

In [ ]:
import mindnlp
#dir(mindnlp.transformers)    

### 1.3 从HuggingFace加载模型到GPU

In [ ]:
from mindnlp.transformers import ViTForImageClassification         
import mindspore as ms        


# 加载预训练的 ViT 模型
model = ViTForImageClassification.from_pretrained('google/vit-base-patch16-224')

#### 1.4 运用 Vit 模型

1. 加载图片；这里使用的是COCO数据集（Common Objects in Context）的2017验证集，这是一个用于物体检测、分割和图像描述等任务的常用基准数据集。COCO数据集包含日常生活中的复杂场景，标注了80种通用物体类别，[COCO数据集](https://cocodataset.org/#download)。

In [ ]:
from PIL import Image
import requests

url = 'http://images.cocodataset.org/val2017/000000000872.jpg'
#url = 'http://images.cocodataset.org/val2017/000000039769.jpg' #如想测试其他图片例子
image = Image.open(requests.get(url, stream=True).raw)
image

2. 使用ViTImageProcessor对图像进行预处理；该模型只接受224x224的输入分辨率，我们使用图像处理器来完成此操作，它负责调整大小和归一化。

In [ ]:
from mindnlp.transformers import ViTImageProcessor

# 加载预训练的 ViT 图像处理器
processor = ViTImageProcessor.from_pretrained('google/vit-base-patch16-224')

# 处理输入图像，返回MindSpore Tensor
inputs = processor(images=image, return_tensors="ms")  # 使用“ms”来生成MindSpore的tensor
pixel_values = inputs['pixel_values']  # 从返回的字典中提取 pixel_values


In [ ]:
print(pixel_values.shape)

3. 进行模型预测；将图像输入到ViT模型中，该模型由类似BERT的编码器和一个线性分类头组成，分类头位于最后一个[CLS]标记的隐藏状态之上。

In [ ]:
from mindspore import ops

# 前向传播进行推理，计算logits
outputs = model(pixel_values)

# 提取logits
logits = outputs['logits']  

# 打印 logits 的形状
logits_shape = logits.shape
print(logits_shape)  

In [ ]:
prediction = logits.argmax(-1)
print("Predicted class:", model.config.id2label[prediction.item()])